In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task4.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task4.csv")
structured_val = pd.read_csv("structured_val_predictions_task4.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,0,0.021105
1,1,0,0.031085
2,2,0,0.023220
3,3,0,0.034456
4,4,1,0.542742


,sample_id,y_true,pred_prob_ecg
0,0,0,0.057976
1,1,0,0.039260
2,2,0,0.033925
3,3,0,0.037469
4,4,1,0.150988


,sample_id,y_true,pred_prob_structured
0,0,0,0.078643
1,1,0,0.055527
2,2,0,0.017421
3,3,0,0.017760
4,4,1,0.641420


18986
18986
18986


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.021105,0.057976,0.078643
1,1,0,0.031085,0.039260,0.055527
2,2,0,0.023220,0.033925,0.017421
3,3,0,0.034456,0.037469,0.017760
4,4,1,0.542742,0.150988,0.641420
...,...,...,...,...,...
18981,18981,0,0.001257,0.039798,0.004710
18982,18982,0,0.020567,0.040688,0.004670
18983,18983,0,0.001461,0.048210,0.031151
18984,18984,0,0.052814,0.047524,0.397949


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task4.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task4.csv")
structured_test = pd.read_csv("structured_test_predictions_task4.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,0,0.013415
1,1,0,0.013274
2,2,0,0.010072
3,3,0,0.038683
4,4,0,0.081593


,sample_id,y_true,pred_prob_ecg
0,0,0,0.242969
1,1,0,0.204618
2,2,0,0.044358
3,3,0,0.067435
4,4,0,0.061398


,sample_id,y_true,pred_prob_structured
0,0,0,0.123264
1,1,0,0.003985
2,2,0,0.018018
3,3,0,0.646426
4,4,0,0.002664


18986
18986
18986


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.013415,0.242969,0.123264
1,1,0,0.013274,0.204618,0.003985
2,2,0,0.010072,0.044358,0.018018
3,3,0,0.038683,0.067435,0.646426
4,4,0,0.081593,0.061398,0.002664
...,...,...,...,...,...
18981,18981,0,0.432893,0.116855,0.100746
18982,18982,0,0.002975,0.093181,0.004644
18983,18983,0,0.010449,0.114281,0.004366
18984,18984,0,0.017935,0.036914,0.047004


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.92264
Logistic multimodal test AUPRC: 0.67565

Top positive coefficients:
        feature     coef
   logit_struct 2.920682
    logit_notes 0.936412
      logit_ecg 0.316894
   ecg_x_struct 0.239149
       prob_std 0.151342
     prob_range 0.115893
 notes_x_struct 0.108473
    notes_x_ecg 0.104554
       prob_max 0.102142
struct_gt_notes 0.078918

Top negative coefficients:
             feature      coef
notes_x_ecg_x_struct -0.053491
        notes_gt_ecg -0.058275
            ecg_conf -0.059513
          notes_conf -0.059789
       struct_gt_ecg -0.060107
    struct_minus_ecg -0.160754
     pred_prob_notes -0.302358
       pred_prob_ecg -0.399023
pred_prob_structured -0.501090
           prob_mean -0.555448
